In [1]:
!pip install tensorboardX rdkit
!pip install rdkit-pypi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 78.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 75.1 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as Data
torch.manual_seed(8) # for reproduce


from sklearn.model_selection import LeaveOneOut
import time
from tqdm import tqdm
import numpy as np
import gc
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/Posdoc/Native_AFP/code/')
sys.setrecursionlimit(50000)
import pickle
torch.backends.cudnn.benchmark = True
torch.set_default_tensor_type('torch.cuda.FloatTensor')
# from tensorboardX import SummaryWriter
torch.nn.Module.dump_patches = True
import copy
import pandas as pd
#then import my own modules
from AttentiveFP import Fingerprint, Fingerprint_viz, save_smiles_dicts, get_smiles_dicts, get_smiles_array, moltosvg_highlight

/usr/local/lib/python3.11/dist-packages/torch/__init__.py:614: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:451.)
  _C._set_default_tensor_type(t)


In [3]:
from rdkit import Chem
# from rdkit.Chem import AllChem
from rdkit.Chem import QED
from rdkit.Chem import rdMolDescriptors, MolSurf
from rdkit.Chem.Draw import SimilarityMaps
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
%matplotlib inline
from numpy.polynomial.polynomial import polyfit
import matplotlib.pyplot as plt
from matplotlib import gridspec
import matplotlib.cm as cm
import matplotlib
import seaborn as sns; sns.set_style("darkgrid")
from IPython.display import SVG, display
import sascorer
import itertools
from sklearn.metrics import r2_score
import scipy


import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
import pickle
import os
import time
from rdkit import Chem
from sklearn.model_selection import LeaveOneOut
from collections import defaultdict
from sklearn.model_selection import ParameterSampler


device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [6]:

def prepare_model_and_data(raw_filename, target_name='Calx', targets=None, random_seed=888, batch_size=50, epochs=15, p_dropout=0.5, fingerprint_dim=280, weight_decay=4.9, learning_rate=3.4, radius=2, T=2, per_target_output_units_num=1):
    if targets is None:
        targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    feature_filename = raw_filename.replace('.csv', '.pickle')
    filename = raw_filename.replace('.csv', '')
    prefix_filename = raw_filename.split('/')[-1].replace('.csv', '')
    smiles_targets_df = pd.read_csv(raw_filename)

    smilesList = smiles_targets_df.SMILES.values
    print("number of all smiles: ", len(smilesList))

    atom_num_dist = []
    remained_smiles = []
    canonical_smiles_list = []
    for smiles in smilesList:
        try:
            mol = Chem.MolFromSmiles(smiles)
            atom_num_dist.append(len(mol.GetAtoms()))
            remained_smiles.append(smiles)
            canonical_smiles_list.append(Chem.MolToSmiles(Chem.MolFromSmiles(smiles), isomericSmiles=True))
        except:
            print(smiles)
            pass
    print("number of successfully processed smiles: ", len(remained_smiles))

    smiles_targets_df = smiles_targets_df[smiles_targets_df["SMILES"].isin(remained_smiles)]
    smiles_targets_df['cano_smiles'] = canonical_smiles_list

    start_time = str(time.ctime()).replace(':', '-').replace(' ', '_')
    output_units_num = len(targets) * per_target_output_units_num

    if os.path.isfile(feature_filename):
        feature_dicts = pickle.load(open(feature_filename, "rb"))
    else:
        feature_dicts = save_smiles_dicts(smilesList, filename)
    feature_dicts = get_smiles_dicts(smilesList)

    remained_df = smiles_targets_df[smiles_targets_df["cano_smiles"].isin(feature_dicts['smiles_to_atom_mask'].keys())]

    x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, smiles_to_rdkit_list = get_smiles_array([canonical_smiles_list[0]], feature_dicts)
    num_atom_features = x_atom.shape[-1]
    num_bond_features = x_bonds.shape[-1]

    loss_function = nn.MSELoss()
    model = Fingerprint(radius, T, num_atom_features, num_bond_features, fingerprint_dim, output_units_num, p_dropout)
    model.to(device)

    optimizer = optim.Adam(model.parameters(), 10**-learning_rate, weight_decay=10**-weight_decay)

    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    #print(params)
    #for name, param in model.named_parameters():
    #    if param.requires_grad:
    #        print(name, param.data.shape)

    return model, optimizer, loss_function, remained_df, targets, feature_dicts


In [7]:
def train(model, dataset, optimizer, loss_function, epoch, batch_size, targets, feature_dicts, ratio_list):
    model.train()
    np.random.seed(epoch)
    
    if len(dataset) <= batch_size:
        batch_list = [dataset.index]
    else:
        valList = np.arange(0, dataset.shape[0])
        np.random.shuffle(valList)
        batch_list = [valList[i:i+batch_size] for i in range(0, dataset.shape[0], batch_size)]
    
    device = next(model.parameters()).device
    total_loss = 0
    
    for counter, batch in enumerate(batch_list):
        batch_df = dataset.loc[batch, :]
        smiles_list = batch_df.cano_smiles.values
        x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array(smiles_list, feature_dicts)
        
        x_atom = torch.Tensor(x_atom).to(device)
        x_bonds = torch.Tensor(x_bonds).to(device)
        x_atom_index = torch.LongTensor(x_atom_index).to(device)
        x_bond_index = torch.LongTensor(x_bond_index).to(device)
        x_mask = torch.Tensor(x_mask).to(device)
        
        atoms_prediction, mol_prediction = model(x_atom, x_bonds, x_atom_index, x_bond_index, x_mask)
        
        optimizer.zero_grad()
        loss = 0.0
        
        for i, target in enumerate(targets):
            y_pred = mol_prediction[:, i]
            y_val = torch.Tensor(batch_df[target].values).squeeze().to(device)
            target_loss = loss_function(y_pred, y_val) * (ratio_list[i] ** 2)
            loss += target_loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(batch_list)


def eval(model, dataset, targets, feature_dicts, batch_size):
    model.eval()
    
    eval_MAE_list = {target: [] for target in targets}
    eval_MSE_list = {target: [] for target in targets}
    y_val_list = {target: [] for target in targets}
    y_pred_list = {target: [] for target in targets}
    smiles_list = []
    
    if len(dataset) <= batch_size:
        batch_list = [dataset.index]
    else:
        valList = np.arange(0, dataset.shape[0])
        batch_list = [valList[i:i+batch_size] for i in range(0, dataset.shape[0], batch_size)]
    
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for batch in batch_list:
            batch_df = dataset.loc[batch, :]
            batch_smiles = batch_df.cano_smiles.values
            smiles_list.extend(batch_smiles)
            
            x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array(batch_smiles, feature_dicts)
            
            x_atom = torch.Tensor(x_atom).to(device)
            x_bonds = torch.Tensor(x_bonds).to(device)
            x_atom_index = torch.LongTensor(x_atom_index).to(device)
            x_bond_index = torch.LongTensor(x_bond_index).to(device)
            x_mask = torch.Tensor(x_mask).to(device)
            
            atoms_prediction, mol_prediction = model(x_atom, x_bonds, x_atom_index, x_bond_index, x_mask)
            
            for i, target in enumerate(targets):
                y_pred = mol_prediction[:, i].view(-1, 1)
                y_val = torch.Tensor(batch_df[target].values.reshape(-1, 1)).to(device)
                
                MAE = F.l1_loss(y_pred, y_val, reduction='none')
                MSE = F.mse_loss(y_pred, y_val, reduction='none')
                
                y_pred_list[target].extend(y_pred.cpu().numpy().flatten())
                y_val_list[target].extend(y_val.cpu().numpy().flatten())
                
                eval_MAE_list[target].extend(MAE.cpu().numpy().flatten())
                eval_MSE_list[target].extend(MSE.cpu().numpy().flatten())
    
    eval_MAE = np.array([np.mean(eval_MAE_list[target]) for target in targets])
    eval_MSE = np.array([np.mean(eval_MSE_list[target]) for target in targets])
    
    return eval_MAE, eval_MSE, y_pred_list, y_val_list, smiles_list




import numpy as np
from collections import defaultdict
from sklearn.model_selection import LeaveOneOut

def train_and_evaluate(model, remained_df, targets, feature_dicts, optimizer, loss_function, batch_size, num_epochs, patience=50, min_delta=0.0001, prefix_filename='', start_time=''):    
    best_param = {"train_epoch": 0, "test_epoch": 0, "train_MSE": float('inf'), "test_MSE": float('inf')}
    stats_list = []
    ratio_list = [1.0] * len(targets)
    loo = LeaveOneOut()
    fold_results = []
    best_fold_results = []
    regression = {}

    train_predictions = {target: [] for target in targets}
    train_actuals = {target: [] for target in targets}
    test_predictions = {target: [] for target in targets}
    test_actuals = {target: [] for target in targets}
    
    initial_state = {k: v.clone() for k, v in model.state_dict().items()}
    for fold, (train_index, test_index) in enumerate(loo.split(remained_df)):
        print(f"\nFold {fold + 1}/{len(remained_df)}")

        # Reset model parameters to initial state
        model.load_state_dict(initial_state)
        
        # Reset optimizer state
        optimizer.state = defaultdict(dict)

        train_df = remained_df.iloc[train_index].copy()
        test_df = remained_df.iloc[test_index].copy()
        current_smile = test_df['Host'].values[0]

        best_train_mse = float('inf')
        best_model_state = None
        best_val_mse = float('inf')
        epochs_no_improve = 0
        best_train_metrics = None

        for epoch in range(num_epochs):
            train_loss = train(model, train_df, optimizer, loss_function, epoch, batch_size, targets, feature_dicts, ratio_list)
            train_MAE, train_MSE, train_pred, train_actual, _ = eval(model, train_df, targets, feature_dicts, batch_size)

            if train_MSE.mean() < best_train_mse:
                best_train_mse = train_MSE.mean()
                best_model_state = model.state_dict()
                best_param["train_epoch"] = epoch
                best_param["train_MSE"] = train_MSE.mean()
                best_train_metrics = (train_MAE, train_MSE, train_pred, train_actual)

            # Evaluate on validation set
            val_MAE, val_MSE, val_pred, val_actual, _ = eval(model, test_df, targets, feature_dicts, batch_size)

            if val_MSE.mean() < best_val_mse - min_delta:
                best_val_mse = val_MSE.mean()
                epochs_no_improve = 0
                best_model_state = model.state_dict()
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

        # Load the best model state for this fold
        model.load_state_dict(best_model_state)

        # Use the best train metrics
        train_MAE, train_MSE, train_pred, train_actual = best_train_metrics

        # Save train metrics only for the best epoch
        for target in targets:
            train_predictions[target].extend(train_pred[target])
            train_actuals[target].extend(train_actual[target])

        # Evaluate on test set only once, after training is complete
        test_MAE, test_MSE, test_pred, test_actual, test_smiles = eval(model, test_df, targets, feature_dicts, batch_size)

        for target in targets:
            test_predictions[target].extend(test_pred[target])
            test_actuals[target].extend(test_actual[target])

        best_fold_results.append((train_MAE.mean(), train_MSE.mean(), test_MAE.mean(), test_MSE.mean()))
        
        # Create the dictionary entry for the current test row
        regression[current_smile] = {}
        for target in targets:
            regression[current_smile][target] = {
                "predicted": test_pred[target][0],
                "actual": test_actual[target][0]
            }

        print(f"\nTest point details for Fold {fold + 1}:")
        print(f"SMILES: {current_smile}")
        for target in targets:
            print(f"{target}:")
            print(f" Actual value: {test_actual[target][0]:.4f}")
            print(f" Predicted value: {test_pred[target][0]:.4f}")

        fold_results.append({
            'fold': fold + 1,
            'smiles': current_smile,
            'actual_values': {target: test_actual[target][0] for target in targets},
            'predicted_values': {target: test_pred[target][0] for target in targets}
        })

    overall_train_mae = np.mean([res[0] for res in best_fold_results])
    overall_train_mse = np.mean([res[1] for res in best_fold_results])
    overall_test_mae = np.mean([res[2] for res in best_fold_results])
    overall_test_mse = np.mean([res[3] for res in best_fold_results])

    print("\nFinal Results:")
    print(f"Overall Training MAE: {overall_train_mae:.4f}")
    print(f"Overall Training MSE: {overall_train_mse:.4f}")
    print(f"Overall Test MAE: {overall_test_mae:.4f}")
    print(f"Overall Test MSE: {overall_test_mse:.4f}")

    return overall_test_mse

In [9]:
def random_search(param_distributions, n_iter):
    param_list = list(ParameterSampler(param_distributions, n_iter=n_iter, random_state=42))
    return param_list

def run_random_search(raw_filename, n_iter=10):
    # Define the hyperparameter space
    param_distributions = {

        'radius': [2, 3, 4, 5, 6],
        'fingerprint_dim': [30, 70, 150, 210, 300],
        'p_dropout': [0.1,0.2,0.3, 0.4, 0.5],
        'weight_decay': [2,3, 4, 5,6],
        'learning_rate': [2, 3, 4, 5]
    }

    # Fixed parameters
    fixed_params = {
        'T': 2,
        'batch_size': 50,
        'epochs': 800
    }

    # Generate random hyperparameter combinations
    param_list = random_search(param_distributions, n_iter)

    best_mse = float('inf')
    best_params = None

    for params in param_list:
        print(f"Testing parameters: {params}")

        # Combine fixed and variable parameters
        current_params = {**fixed_params, **params}

        # Prepare model and data with current hyperparameters
        model, optimizer, loss_function, remained_df, targets, feature_dicts = prepare_model_and_data(
            raw_filename, **current_params
        )

        # Train and evaluate the model
        mse = train_and_evaluate(
            model, remained_df, targets, feature_dicts, optimizer, loss_function,
            fixed_params['batch_size'], fixed_params['epochs']
        )

        print(f"MSE: {mse}")

        if mse < best_mse:
            best_mse = mse
            best_params = params

    print(f"Best parameters: {best_params}")
    print(f"Best MSE: {best_mse}")

    return best_params, best_mse

if __name__ == "__main__":
    raw_filename = "/notebooks/data/cal_abs.csv"
    best_params, best_mse = run_random_search(raw_filename, n_iter=10)

Testing parameters: {'weight_decay': 2, 'radius': 4, 'p_dropout': 0.5, 'learning_rate': 4, 'fingerprint_dim': 70}
number of all smiles:  40
number of successfully processed smiles:  40

Fold 1/40
Early stopping at epoch 257

Test point details for Fold 1:
SMILES: AP8
H3K4:
 Actual value: -1.3093
 Predicted value: 0.9829
H3K4ac:
 Actual value: 0.0000
 Predicted value: 1.4859
H3K4me1:
 Actual value: -3.0791
 Predicted value: -0.0868
H3K4me2:
 Actual value: -5.1850
 Predicted value: -1.6691
H3K4me3:
 Actual value: -5.0207
 Predicted value: -1.9796
H3K9me3:
 Actual value: -3.7723
 Predicted value: -1.0789
H3R2me2a:
 Actual value: -4.1997
 Predicted value: -0.5341
H3R2me2s:
 Actual value: -2.6037
 Predicted value: 0.1323

Fold 2/40
Early stopping at epoch 301

Test point details for Fold 2:
SMILES: AH4
H3K4:
 Actual value: -0.1165
 Predicted value: 0.8921
H3K4ac:
 Actual value: 1.4110
 Predicted value: 1.3513
H3K4me1:
 Actual value: -2.2073
 Predicted value: -0.0280
H3K4me2:
 Actual value: 

In [ ]:
def visualize_results(regression_dict):
    # Prepare data
    all_targets = set()
    for smile_data in regression_dict.values():
        all_targets.update(smile_data.keys())
    all_targets = sorted(list(all_targets))

    # Create a figure with subplots for each target
    n_targets = len(all_targets)
    n_cols = 3
    n_rows = (n_targets + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
    fig.suptitle("Predicted vs Actual Values for Each Target", fontsize=16)

    # Flatten axes array for easy indexing
    axes = axes.flatten()

    for i, target in enumerate(all_targets):
        predicted = []
        actual = []
        for smile_data in regression_dict.values():
            if target in smile_data:
                pred, act = smile_data[target]
                predicted.append(pred)
                actual.append(act)
        
        ax = axes[i]
        ax.scatter(actual, predicted, alpha=0.5)
        ax.set_title(target)
        ax.set_xlabel("Actual")
        ax.set_ylabel("Predicted")
        
        # Add diagonal line
        lims = [
            np.min([ax.get_xlim(), ax.get_ylim()]),
            np.max([ax.get_xlim(), ax.get_ylim()]),
        ]
        ax.plot(lims, lims, 'r--', alpha=0.75, zorder=0)
        
        # Set aspect of axis scaling to be equal
        ax.set_aspect('equal', adjustable='box')

    # Remove any unused subplots
    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

# Call the function with your regression dictionary
visualize_results(regression)